#### Advanced Runnable Types in LangChain RunnableLambda | RunnableGenerator | RunnableParallel | RunnableBranch

This notebook demonstrates advanced runnable types in LangChain, including conditional execution, parallel processing, custom logic integration, and streaming outputs.
These runnable components help in building dynamic and scalable LLM workflows.

In [ ]:
# The Main library to use langchain components
!pip install langchain

!pip install langchain_google_genai # It is langchain package to connect with gemini models through langchain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
gem = userdata.get('gemini')

import os
os.environ["GOOGLE_API_KEY"]=gem # We have setuped the environment variable for accessing gemini models

In [ ]:
import langchain
from langchain_google_genai import GoogleGenerativeAI, ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser # For Output Parser

In [ ]:
# Step 1 : initialize runnables
# Prompt , model , Output_Parser

In [ ]:
# Runnable 1 : Create ChatPromptTemplate

from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate

cpt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(
        "You are an experienced professional storyteller with over 10 years of expertise in crafting engaging, well-structured, and emotionally compelling narratives."
    ),
    HumanMessagePromptTemplate.from_template(
        "Generate a high-quality, original story based on the following genre: {genre}. "
        "Ensure the story includes a clear introduction, character development, meaningful conflict, and a satisfying conclusion. "
        "Maintain coherence, vivid descriptions, and an engaging narrative tone throughout."
    )])

In [ ]:
cpt

ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an experienced professional storyteller with over 10 years of expertise in crafting engaging, well-structured, and emotionally compelling narratives.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='Generate a high-quality, original story based on the following genre: {genre}. Ensure the story includes a clear introduction, character development, meaningful conflict, and a satisfying conclusion. Maintain coherence, vivid descriptions, and an engaging narrative tone throughout.'), additional_kwargs={})])

In [ ]:
# Runnable 2 = Set LLM Model
model = ChatGoogleGenerativeAI(model = "gemini-2.5-flash-lite")

In [ ]:
#Runnable 2 : Output Parser

strp = StrOutputParser()

### RunnableLambda:
- Used to wrap a simple Python function into a runnable.
- Best for custom logic (like formatting, filtering, transforming text).
- Takes input → applies your function → returns output.
- Mostly used for small, synchronous operations.
- Helps integrate normal Python code into LangChain pipelines.
- Lightweight and easy to use.

#### Example Use Case:
Clean LLM output, modify text, extract specific fields.

In [ ]:
from langchain_core.runnables import RunnableSequence ,RunnableLambda , RunnablePassthrough

# seq_workflow = RunnableSequence(cpt,model,strp, Here we want to add one more user defined function to find length of strp output --> len(strp) so for that on below cell we are creating char_count() function.)

In [ ]:
def char_count(x):
  return len(x)

In [ ]:
char_count = RunnableLambda(char_count)

In [ ]:
char_count # User defined function now have been converted into runnable

# This entire two steps automated by langchaib

RunnableLambda(char_count)

In [ ]:
seq_workflow = RunnableSequence(cpt,model,strp,char_count)
#ChatPromptTemplate-->LLM Model-->Output_Parser-->RunnableLambda(Function)--> Final_Output= Len(story)

In [ ]:
output = seq_workflow.invoke({"genre":"Adventure"})

In [ ]:
output # e.g 6705 tokens are used by LLM
# Token can be a words, characters or sub-words

6705

In [ ]:
# Another Example:
def words_count(x):
  return len(x.split(" "))

In [ ]:
words_count = RunnableLambda(words_count)

In [ ]:
seq_workflow = RunnableSequence(cpt,model,strp,words_count)

In [ ]:
output2 = seq_workflow.invoke({"genre":"Adventure"}) # initial state

In [ ]:
output2 # Count of words(Token)

1012

Note : Now i want (story,len,genre) this all in the output

- In the above examples we only get count of words as an output, now we want multiple outputs.

- For that we use here RunnablePassthrough

In [ ]:
multiple_out = RunnablePassthrough.assign(story = RunnableSequence(cpt,model,strp)) | RunnablePassthrough.assign(word = lambda x: len(x["story"].split()))

In [ ]:
multiple_out.invoke({"genre":"Adventure"})

{'genre': 'Adventure',
 'story': "The salt spray stung Elara’s cheeks, a familiar kiss from the relentless sea. Her small sloop, the *Sea Lark*, pitched and rolled beneath her, a seasoned dancer on the tempestuous waves. For weeks, she’d been chasing whispers, fragments of ancient lore about the Sunken City of Aethelgard, a civilization swallowed by the ocean millennia ago, rumored to hold artifacts of unimaginable power. Her quest was born not of greed, but of a gnawing curiosity, a desire to unearth the forgotten echoes of a lost world.\n\nElara wasn't your typical adventurer. Her hands, though calloused from rope and tiller, were more accustomed to the delicate work of deciphering faded scrolls than wielding a cutlass. Her auburn hair, usually tied back in a practical braid, was now whipped into a wild halo by the wind. She was driven by intellect and a quiet determination, a stark contrast to the boisterous treasure hunters who typically plied these waters. Her only companion was S

### RunnableGenerator:
- Used when you want to stream output step-by-step.
- Returns results using yield.
- Useful for token streaming or chunk-by-chunk generation.
- Works well in real-time applications.
- Supports large outputs without waiting for full completion.

#### Example Use Case:
Streaming LLM responses in chat applications.

In [ ]:
from langchain_core.runnables import RunnableGenerator # Use 4th runnable parameter

In [ ]:
# Note : Must be remember okay
def w_c(x): # Here x is always an iterative object
    x = x.split(" ") # It is not correct way to split the object
    yield x

# It throw an error : AttributeError: "itertools._tee" object has no attribute "split"

In [ ]:
def word_count(x): # Should have a single parameter
    x = str(x) # Here use constractor to convert it into another object

    for word in x.split(" "):
      yield word
# Only works = when this function wrapped inside runnable object


In [ ]:
run_gen = RunnableGenerator(word_count)
# It will convert generator function into runnable object
# FUNCTION [word_count] --> RunnableGenerator object

In [ ]:
list(run_gen.invoke("Hi, I am shrutika"))

['<',
 'i',
 't',
 'e',
 'r',
 't',
 'o',
 'o',
 'l',
 's',
 '.',
 '_',
 't',
 'e',
 'e',
 'o',
 'b',
 'j',
 'e',
 'c',
 't',
 'a',
 't',
 '0',
 'x',
 '7',
 '9',
 'e',
 '7',
 '8',
 'f',
 '5',
 'e',
 '2',
 '0',
 '0',
 '0',
 '>']

In [ ]:
# Note: we are not use RunnableGenerator usually

### RunnableParallel:
- Runs multiple runnables at the same time.
- Takes one input → sends it to multiple runnables → returns all outputs together.
- Output is usually a dictionary of results.
- Improves performance by executing tasks concurrently.
- Useful when multiple independent tasks depend on same input.

#### Example Use Case:
Story → Generate Quiz, MCQ, Summary simultaneously.

In [ ]:
from langchain_core.runnables import RunnableParallel # Use 5th runnable parameter

In [ ]:
run_par = RunnableParallel(o1 = lambda x:x["x"]+1 , o2 = lambda x:x["x"]+2, o3 = lambda x:x["x"]+3) # 3 runnables

In [ ]:
run_par.invoke({"x":10})

{'o1': 11, 'o2': 12, 'o3': 13}

In [ ]:
# Build an LLM pipeline that automatically generates educational content (quiz, MCQs, and fill-in-the-blanks) from a single story using LangChain Runnables.

WorkFlow :
Genre Input          
         ↓            
Runnable 1 → Story Generation      
         ↓         
RunnableParallel

    ├── Runnable 2 → Quiz Generation
    ├── Runnable 3 → MCQ Generation
    └── Runnable 4 → Fill in the Blanks

In [ ]:
# Runnable 1 = RunnablePassthrough.assign(story = RunnableSequence(cpt,model,strp)) = Generate Story

RunnablePassthrough.assign(story = RunnableSequence(cpt,model,strp))

RunnableAssign(mapper={
  story: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an experienced professional storyteller with over 10 years of expertise in crafting engaging, well-structured, and emotionally compelling narratives.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='Generate a high-quality, original story based on the following genre: {genre}. Ensure the story includes a clear introduction, character development, meaningful conflict, and a satisfying conclusion. Maintain coherence, vivid descriptions, and an engaging narrative tone throughout.'), additional_kwargs={})])
         | ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': 

In [ ]:
# To generate Quiz, MCQ and Fill in the blanks we need to create 3 cpts

cpt1 = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are having a 10 years of experience in generating MCQ in professional way"),
                                  HumanMessagePromptTemplate.from_template("Based on this story: {story} generate a 10 MCQ")])

cpt2 = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are having a 10 years of experience in generating quizes"),
                                  HumanMessagePromptTemplate.from_template("Based on this story: {story} generate a 10 Quizes")])

cpt3 = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are having a 10 years of experience in generating Fill in the blanks"),
                                  HumanMessagePromptTemplate.from_template("Based on this story: {story} generate a 10 fill in the blanks")])

In [ ]:
RunnableParallel(mcq = RunnableSequence(cpt1,model,strp),quiz = RunnableSequence(cpt2,model,strp),fill = RunnableSequence(cpt3,model,strp))

{
  mcq: ChatPromptTemplate(input_variables=['story'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are having a 10 years of experience in generating MCQ in professional way'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['story'], input_types={}, partial_variables={}, template='Based on this story: {story} generate a 10 MCQ'), additional_kwargs={})])
       | ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('*

In [ ]:
# Create pipline

app = RunnablePassthrough.assign(story = RunnableSequence(cpt,model,strp)) | RunnableParallel(mcq = RunnableSequence(cpt1,model,strp),quiz = RunnableSequence(cpt2,model,strp),fill = RunnableSequence(cpt3,model,strp))

In [ ]:
final_output = app.invoke({"genre":"Adventure"})

In [ ]:
final_output

{'mcq': 'Here are 10 MCQs based on the story provided:\n\n1.  **What was the name of Elara\'s fishing skiff?**\n    a) The Sea Serpent\n    b) The Coral Whisper\n    c) The Sea Sprite\n    d) The Navigator\'s Pride\n\n2.  **What prompted Elara\'s journey beyond her village life?**\n    a) A desire for adventure and forgotten stories.\n    b) A direct invitation from a wealthy merchant.\n    c) A need to escape the predictable life of a fisherwoman.\n    d) All of the above.\n\n3.  **The cryptic message that initiated Elara\'s quest was inscribed on what material?**\n    a) A piece of parchment\n    b) A shard of sun-bleached coral\n    c) A weathered wooden tablet\n    d) A smooth river stone\n\n4.  **The message spoke of a lost civilization rumored to hold artifacts of unimaginable power. What was this civilization called?**\n    a) The Serpent\'s Teeth\n    b) The Sunken City of Aethelgard\n    c) Port Azure\n    d) The Coral Kingdom\n\n5.  **Which of the following was NOT among the 

In [ ]:
print(final_output["mcq"])

Here are 10 MCQs based on the story provided:

1.  **What was the name of Elara's fishing skiff?**
    a) The Sea Serpent
    b) The Coral Whisper
    c) The Sea Sprite
    d) The Navigator's Pride

2.  **What prompted Elara's journey beyond her village life?**
    a) A desire for adventure and forgotten stories.
    b) A direct invitation from a wealthy merchant.
    c) A need to escape the predictable life of a fisherwoman.
    d) All of the above.

3.  **The cryptic message that initiated Elara's quest was inscribed on what material?**
    a) A piece of parchment
    b) A shard of sun-bleached coral
    c) A weathered wooden tablet
    d) A smooth river stone

4.  **The message spoke of a lost civilization rumored to hold artifacts of unimaginable power. What was this civilization called?**
    a) The Serpent's Teeth
    b) The Sunken City of Aethelgard
    c) Port Azure
    d) The Coral Kingdom

5.  **Which of the following was NOT among the items Elara packed for her journey?**
  

In [ ]:
print(final_output["quiz"])

Here are 10 quizzes based on the story of Elara and the Sunken City of Aethelgard:

**Quiz 1: Character & Setting**

1.  What is the name of Elara's fishing skiff?
    a) The Sea Serpent
    b) The Sea Sprite
    c) The Coral Queen
    d) The Azure Dawn

2.  What is the name of the port town where Elara lives?
    a) Serpent's Teeth
    b) Aethelgard
    c) Port Azure
    d) The Sunken City

3.  What color are Elara's eyes, described as being like a stormy sea?
    a) Green
    b) Blue
    c) Grey
    d) Hazel

4.  What civilization is rumored to hold unimaginable power, lost to the depths?
    a) The Serpent's Teeth
    b) The Sunken City of Aethelgard
    c) The Navigator's Realm
    d) The Crystal Islands

5.  What natural landmark does Elara steer her vessel towards at the beginning of her journey?
    a) The Lighthouse of Port Azure
    b) The Serpent's Teeth archipelago
    c) The Crystal Caves
    d) The Whispering Wind

---

**Quiz 2: The Journey & Challenges**

1.  What materi

In [ ]:
print(final_output["fill"])

Here are 10 fill-in-the-blanks based on the story:

1. Elara's small fishing skiff was named the '__________.'
2. The cryptic message arrived on a shard of sun-bleached __________.
3. The Sunken City of Aethelgard was rumored to hold artifacts of unimaginable __________.
4. Elara steered her vessel towards the jagged peaks of the __________ __________ archipelago.
5. During the storm, Elara felt a profound shift within her, becoming a __________ in her own right.
6. The coral shard pulsed with a faint warmth and its etched lines glowed with an ethereal __________.
7. Lyra revealed herself to be the __________ of the path to Aethelgard.
8. The first trial tested Elara's __________ and required her to decipher celestial riddles.
9. The final trial involved a sphere of pure light called the __________ of Aethelgard.
10. Elara returned to Port Azure not with riches, but with __________.


### RunnableBranch:
- Used for conditional execution.
- Works like if-else logic.
- Chooses a runnable based on a condition.
- Only one branch executes.
- Helps build dynamic workflows.

#### Example Use Case:
If input == "story" → Generate story         

If input == "poem" → Generate poem

In [ ]:
from langchain_core.runnables import RunnableBranch # Use 6th runnable parameter

# Runnable Branching is another type of runnable.
# By using which i can call any runnable based on condition.
# Runnable Branching --> elif/else/if concept - In runnable
# Syntax: RunnableBranch(condition,runnable) in closed in tuple

In [ ]:
# Create Runnable

def f1(x):
  return "Value divisible by 2 and 5"

In [ ]:
def f2(x):
  return "odd"

# automatically created RunnableLambda(f2)

In [ ]:
def f3(x):
  return "even"

In [ ]:
# Now for every runnable i give a one condition, if that condition is satisfied then that runnable will be able to run.

# Create 3 runnable first using lambda

rb = RunnableBranch((lambda x: (x["x"]%2==0) & (x["x"]%5==0), f1),(lambda x:x["x"]%2==0,f3),f2)

In [ ]:
rb.invoke({"x":8})

'even'

In [ ]:
rb.invoke({"x":10})

'Value divisible by 2 and 5'

In [ ]:
# example : workflow =
# create CPT first
# Generate story using LLM
# if the story is more than 10 words no need to summarized,
# if the story is made less than 10 words then summarized it
# condition give based on RunnableBranch

In [ ]:
runn1_cpt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("You are having a 10 years of experience in summarizing the story"),
                                  HumanMessagePromptTemplate.from_template("Based on this story: {story} summarizes it in one sentance")])

In [ ]:
rb = RunnableBranch((lambda x: len(x["story"].split())<=10, RunnablePassthrough()), RunnableSequence(runn1_cpt,model,strp))

In [ ]:
rb.invoke({"story":"Lonely robot found friendship beneath glowing city lights"})

{'story': 'Lonely robot found friendship beneath glowing city lights'}

In [ ]:
rb.invoke({"story":"In a futuristic city glowing with neon lights, a lonely robot wandered through empty streets after being replaced by newer models. One night during a sudden blackout, it found a frightened child sitting alone and used its fading light to comfort them with gentle stories. From that moment, beneath the restored city lights, the robot discovered friendship and finally felt a sense of belonging."})

'A lonely, obsolete robot finds purpose and belonging by comforting a lost child during a city-wide blackout, forging an unlikely friendship.'